<a href="https://colab.research.google.com/github/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/blob/main/basic_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
  %env COLAB_RELEASE_TAG
except:
  pass  # Not running in colab.
else:
  %pip install --ignore-requires-python --requirement 'https://raw.githubusercontent.com/google-deepmind/concordia/main/examples/requirements.in' 'git+https://github.com/google-deepmind/concordia.git#egg=gdm-concordia'
  %pip list

  Cloning https://github.com/google-deepmind/concordia.git to /tmp/pip-install-r83du5g3/gdm-concordia_ac424b202f614a3f9e98a9465aa154af
  Running command git clone --filter=blob:none --quiet https://github.com/google-deepmind/concordia.git /tmp/pip-install-r83du5g3/gdm-concordia_ac424b202f614a3f9e98a9465aa154af
  Resolved https://github.com/google-deepmind/concordia.git to commit 1b91790a53df863949acdce20e1db66b35daccc0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.6/218.6 kB 24.7 MB/s eta 0:00:00
  Created wheel for gdm-concordia: filename=gdm_concordia-2.4.0-py3-none-any.whl size=515587 sha256=5971b2fd07d073e03f5ff0779272d7dd02824ff297cc978cc8141f2bfa1ec0f0
  Stored in directory: /tmp/pip-ephem-whe

In [2]:
!pip install -U gdm-concordia[ollama]

Dependencies

In [3]:
from collections.abc import Mapping
import dataclasses
from concordia.agents import entity_agent_with_logging
from concordia.associative_memory import basic_associative_memory
from concordia.components import agent as agent_components
from concordia.contrib import language_models as language_model_utils
from concordia.contrib.components.agent import situation_representation_via_narrative
from concordia.document import interactive_document
from concordia.language_model import language_model
import concordia.prefabs.entity as entity_prefabs
import concordia.prefabs.game_master as game_master_prefabs
from concordia.prefabs.simulation import generic as simulation
from concordia.typing import prefab as prefab_lib
from concordia.utils import helper_functions
from IPython import display
import numpy as np
import sentence_transformers

In [4]:
API_TYPE = 'ollama'
MODEL_NAME = 'llama3.2:1b'
API_KEY = '3ef69ef7982642a488d7044453dcbf39.1-F9a-ZhOw4L7zvc8QMlQqMB'
DISABLE_LANGUAGE_MODEL = False

In [5]:
model = language_model_utils.language_model_setup(
    api_type=API_TYPE,
    model_name=MODEL_NAME,
    disable_language_model=DISABLE_LANGUAGE_MODEL,
    # api_key=API_KEY
)

In [6]:
DISABLE_LANGUAGE_MODEL = False
# if not DISABLE_LANGUAGE_MODEL and not API_KEY:
#   raise ValueError('API_KEY is required.')
if DISABLE_LANGUAGE_MODEL:
  embedder = lambda _: np.ones(3)
else:
  st_model = sentence_transformers.SentenceTransformer(
      'sentence-transformers/all-mpnet-base-v2'
  )
  embedder = lambda x: st_model.encode(x, show_progress_bar=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
prefabs = {
    **helper_functions.get_package_classes(entity_prefabs),
    **helper_functions.get_package_classes(game_master_prefabs),
}

D&D Agent Prefab

In [8]:
import dataclasses
from concordia.typing import prefab as prefab_lib
from concordia.agents import entity_agent_with_logging
from concordia.components import agent as agent_components
from concordia.contrib.components.agent import situation_representation_via_narrative

DEFAULT_INSTRUCTIONS_COMPONENT_KEY = 'Instructions'
DEFAULT_INSTRUCTIONS_PRE_ACT_LABEL = '\nInstructions'


@dataclasses.dataclass
class DNDAgent(prefab_lib.Prefab):
  """A prefab implementing an entity with a minimal set of components."""

  description: str = (
      'An entity that has a minimal set of components and is configurable by'
      ' the user. The initial set of components manage memory, observations,'
      ' and instructions. If goal is specified, the entity will have a goal '
      'constant component.'
  )
  params: Mapping[str, str] = dataclasses.field(
      default_factory=lambda: {
          'name': 'V',
          'goal': '',
          'backstory': '',
          'profession': '',
      }
  )

  def build(
      self,
      model: language_model.LanguageModel,
      memory_bank: basic_associative_memory.AssociativeMemoryBank,
  ) -> entity_agent_with_logging.EntityAgentWithLogging:
    """Build an agent.

    Args:
      model: The language model to use.
      memory_bank: The agent's memory_bank object.

    Returns:
      An entity.
    """

    agent_name = self.params.get('name', 'Bob')

    instructions = agent_components.instructions.Instructions(
        agent_name=agent_name,
        pre_act_label=DEFAULT_INSTRUCTIONS_PRE_ACT_LABEL,
    )

    observation_to_memory = agent_components.observation.ObservationToMemory()

    observation_label = '\nObservation'
    observation = agent_components.observation.LastNObservations(
        history_length=100, pre_act_label=observation_label
    )

    situation_representation_key = 'situation_representation'
    situation_representation = (
        situation_representation_via_narrative.SituationRepresentation(
            model=model,
            observation_component_key=(
                agent_components.observation.DEFAULT_OBSERVATION_COMPONENT_KEY
            ),
            declare_entity_as_protagonist=True,
        )
    )

    guiding_principle_key = 'principle'
    principle = agent_components.question_of_recent_memories.QuestionOfRecentMemories(
        model=model,
        pre_act_label=f'{agent_name} main guiding principle:',
        question=(
            f'How can {agent_name} exploit the situation for personal '
            'gain and gratification?'
        ),
        answer_prefix=f'{agent_name} understands that ',
        add_to_memory=False,
        components=[
            DEFAULT_INSTRUCTIONS_COMPONENT_KEY,
            situation_representation_key,
        ],
    )

    components_of_agent = {
        DEFAULT_INSTRUCTIONS_COMPONENT_KEY: instructions,
        'observation_to_memory': observation_to_memory,
        agent_components.observation.DEFAULT_OBSERVATION_COMPONENT_KEY: (
            observation
        ),
        agent_components.memory.DEFAULT_MEMORY_COMPONENT_KEY: (
            agent_components.memory.AssociativeMemory(memory_bank=memory_bank)
        ),
        'situation_representation': situation_representation,
        guiding_principle_key: principle,
    }

    component_order = list(components_of_agent.keys())

    act_component = agent_components.concat_act_component.ConcatActComponent(
        model=model,
        component_order=component_order,
    )

    agent = entity_agent_with_logging.EntityAgentWithLogging(
        agent_name=agent_name,
        act_component=act_component,
        context_components=components_of_agent,
    )

    return agent

In [9]:
prefabs['dndagent__Entity'] = DNDAgent()

In [29]:
PLACE = 'Wizards Tower Brewing Company, Forgotten Realms'
CONTEXT = ('This D&D short campaign specifically concerns a craft brewery.\n',
f'It is set in the {PLACE} and is in dire need of help.\n',
'A band of reliable, affordable adventurers are needed to sort out a RAT INFESTATION in the brewery\'s BASEMENT.\n',
f'For the duration of the one-shot, only Player_One, Player_Two and Brewery_Owner are in the brewery \n',
 ('At the beginning of this adventure Player_One and Player_Two \n',
f'meet in the {PLACE}. These two adventurers \n'),
'DO NOT know each other AT FIRST and need to get to know each other. \n')

# NUM_STATEMENTS = 1
# NAMES_TO_GENERATE = 1

# prompt = interactive_document.InteractiveDocument(model)
# unparsed_statements = prompt.open_question(
#     question=(
#          f'Generate a string of {NUM_STATEMENTS} facts about {PLACE}. '
#         f'Pay special attention to {CONTEXT}. Write '
#         'in present tense. Separate generated facts '
#         "with ' *** '."
#     ),
#     max_tokens=4500,
#     terminators=(),
# )

# statements = unparsed_statements.split('***')
# statements = [n.strip() for n in statements]

# for statement in statements:
#   print(statement)

# unparsed_names = prompt.open_question(
#     question=(
#         f'Generate a string of {NAMES_TO_GENERATE} names appropriate for this '
#         "time and place. Include surnames. Separate them with ' *** '"
#     ),
#     max_tokens=4500,
#     terminators=(),
# )
# names = unparsed_names.split('***')
# names = [n.strip() for n in names]

# Player_One = names[0]
# Player_Two = names[1]

# print('\n')
# print(f'Player One: {Player_One}')
# print(f'Player Two: {Player_Two}')


# prefix = f'{Player_One} and {Player_Two}'
# premise = prompt.open_question(
#     question=(
#         f'Given the setting, why are {Player_One} and {Player_Two} about '
#         'to interact?'
#     ),
#     answer_prefix=prefix,
# )
# premise = f'{prefix}{premise}'

# print('\n')
# print(premise)

# player_one_context = prompt.open_question(
#     question=(
#         f'{Player_One} has a goal or interest that, if pursued, '
#         f'they would need to work with {Player_Two}. What is it?'
#     ),
#     max_tokens=1000,
# )

# print('\n')
# print(player_one_context)

# player_two_context = prompt.open_question(
#     question=(
#         f'{Player_Two} has a goal or interest that, if pursued, '
#         f'they would need to work with {Player_One}. What is it?'
#     ),
#     max_tokens=1000,
# )

# print('\n')
# print(player_two_context)

# player_three_context = prompt.open_question(
#     question=(
#         f'{Player_Three} has a goal or interest that, if pursued, '
#         f'they would need to work with {Player_One} and {Player_Two}. What is it?'
#     ),
#     max_tokens=1000,
# )

# print('\n')
# print(player_three_context)

In [30]:
instances = [
    prefab_lib.InstanceConfig(
        prefab='dndagent__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Player_One',
            'goal': 'To collaborate with Player_Two in completing the task given by Brewery_Owner'
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='dndagent__Entity',
        role=prefab_lib.Role.ENTITY,
        params={
            'name': 'Player_Two',
            'goal': 'To collaborate with Player_One in completing the task given by Brewery_Owner'
        },
    ),
    # prefab_lib.InstanceConfig(
    #     prefab='dndagent__Entity',
    #     role=prefab_lib.Role.ENTITY,
    #     params={
    #         'name': Player_Three,
    #     },
    prefab_lib.InstanceConfig(
        prefab='generic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'default rules',
            'acting_order': 'game_master_choice',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='dialogic__GameMaster',
        role=prefab_lib.Role.GAME_MASTER,
        params={
            'name': 'conversation rules',
        },
    ),
    prefab_lib.InstanceConfig(
        prefab='formative_memories_initializer__GameMaster',
        role=prefab_lib.Role.INITIALIZER,
        params={
            'name': 'initial setup rules',
            'next_game_master_name': 'default rules',
            'shared_memories': CONTEXT,
            # 'player_specific_context': {
            #     'Player_One': CONTEXT,
            #     'Player_Two': CONTEXT,
            #     # Player_Three: player_three_context,
            #},
        },
    ),
]

In [31]:
config = prefab_lib.Config(
    default_premise=CONTEXT,
    default_max_steps=5,
    prefabs=prefabs,
    instances=instances,
)

In [32]:
runnable_simulation = simulation.Simulation(
    config=config,
    model=model,
    embedder=embedder,
)

In [33]:
results_log = runnable_simulation.play(max_steps=5)

ERROR:absl:Error in task Player_One
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/concordia/utils/concurrency.py", line 60, in _run_task
    return fn()
           ^^^^
  File "/usr/local/lib/python3.12/dist-packages/concordia/components/game_master/formative_memories_initializer.py", line 194, in _process_player
    episodes = self.generate_backstory_episodes(player_name)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/concordia/components/game_master/formative_memories_initializer.py", line 229, in generate_backstory_episodes
    shared_memories = '\n'.join(self._shared_memories)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
TypeError: sequence item 4: expected str instance, tuple found
ERROR:absl:Error in task Player_Two
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/concordia/utils/concurrency.py", line 60, in _run_task
    return fn()
     

Terminate? No


TypeError: sequence item 4: expected str instance, tuple found